In [1]:
import glob
import os
import pandas as pd

In [2]:
base_path = 'shadow/examples/docs/tor-rep-exit/shadow.data'
data = []

# Match all files in the shadow.data directory
files = glob.glob(os.path.join(base_path, 'hosts/**/tor.*.stdout'), recursive=True)
    
for file in files:
    # This split gets 'exit_malicious'
    host_name = file.split(os.sep)[-2]
    
    with open(file, 'r', errors='ignore') as f:
        for line in f:
            data.append({'host': host_name, 'message': line.strip()})

df = pd.DataFrame(data)

In [3]:
# Count successes grouped by host
host_success_counts = df[df['message'].str.contains("FFS-ZKP handshake successful", na=False)] \
                        .groupby('host')['message'].count()

print("Handshake Successes per Exist Node")
print(host_success_counts)

Handshake Successes per Exist Node
host
exit1       1
exit2    2519
exit3    1132
exit4    1501
Name: message, dtype: int64


In [5]:
# Replay Attack (Attack 2)
malicious_host = "exit1"

replay_detected = df[(df['host'] == malicious_host) & (df['message'].str.contains(r"Executing Replay", na=False))].shape[0]

replay_caught = df[(df['host'] == malicious_host) & (df['message'].str.contains(r"Replaying Detected", na=False))].shape[0]

successful_handshakes = df[df['host'] == malicious_host]['message'].str.contains(r"FFS-ZKP handshake successful", na=False).sum()

print(f"Replay attempts through Malicious {malicious_host}: {replay_detected}")
print(f"Replay attempts caught: {replay_caught}")
print(f"Replay attempts that PASSED: {successful_handshakes}\n")

if successful_handshakes == 1:
    print(f"SUCCESS: Zero false negatives for Replay Attacks. All duplicate attempts were blocked")
else:
    print(f"WARNING: {successful_handshakes} replay attempts were not successfully rejected")

Replay attempts through Malicious exit1: 4844
Replay attempts caught: 4843
Replay attempts that PASSED: 1

SUCCESS: Zero false negatives for Replay Attacks. All duplicate attempts were blocked
